# 🤖 Actividad: Interfaz Multimodal con Ollama y Gradio

---

## 📋 Información General

| Campo | Detalle |
|---|---|
| **Asignatura** | Inteligencia Artificial / Modelos de Lenguaje |
| **Duración estimada** | 90 – 120 minutos |
| **Nivel** | Intermedio |
| **Herramientas** | Google Colab, Ollama, Gradio, Ngrok |

---

## 🎯 Objetivo de Aprendizaje

Al finalizar esta actividad, el estudiante será capaz de:

1. **Levantar modelos LLM localmente** usando Ollama dentro de Google Colab.
2. **Distinguir** entre un modelo de generación de texto y uno de generación de imágenes.
3. **Construir una interfaz interactiva** con Gradio que permita seleccionar y usar ambos modelos.
4. **Publicar** la aplicación en Internet usando un túnel Ngrok.

---

## 🗺️ Mapa de la Actividad

```
PARTE 1 ──► Instalación y configuración del entorno
PARTE 2 ──► Levantar Ollama + modelo de TEXTO (TinyLlama)
PARTE 3 ──► Levantar modelo de IMAGEN (Stable Diffusion)
PARTE 4 ──► Construir interfaz Gradio unificada
PARTE 5 ──► Reflexión
PARTE 6 ──► Retos opcionales
PARTE 7 ──► Publicar con Ngrok
```

> ⚠️ **Antes de empezar:** Activa la GPU T4 en `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.
>
> ▶️ **Cómo usar este notebook:** Ejecuta cada celda **en orden**, de arriba hacia abajo. No saltes celdas.

---
## 🔧 PARTE 1 — Instalación del Entorno

Primero instalamos todas las herramientas de Python que usaremos a lo largo de la actividad.

| Librería | ¿Para qué sirve? |
|---|---|
| `gradio` | Crear interfaces web interactivas con Python |
| `requests` | Hacer peticiones HTTP al servidor Ollama |
| `pillow` | Manipular y mostrar imágenes |
| `httpx` | Cliente HTTP moderno (requerido por Gradio) |

In [ ]:
# ============================================================
# CELDA 1 — Instalar librerías de Python
# ============================================================

# El símbolo ! le dice a Colab que ejecute un comando del sistema,
# no de Python. 'pip' es el gestor de paquetes de Python.
# La opción -q (quiet) reduce el texto de salida en pantalla.
!pip install -q gradio requests pillow httpx

# Importamos gradio para confirmar que la instalación fue exitosa
import gradio as gr

# Imprimimos la versión instalada como verificación
print(f'✅ Gradio versión: {gr.__version__}')
print('✅ Todas las librerías instaladas correctamente')

In [ ]:
# ============================================================
# CELDA 2 — Instalar Ollama en el sistema operativo
# ============================================================

# Ollama es un servidor que descarga y gestiona modelos LLM localmente.
# Se instala a nivel del sistema (no de Python), por eso usamos curl.
# curl descarga el script de instalación oficial, y sh lo ejecuta.
# Flags de curl: -f (falla silenciosamente), -s (silencioso),
#                -S (muestra errores), -L (sigue redirecciones)
import subprocess, time

print('📥 Instalando Ollama en el sistema...')
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama instalado correctamente')

---
## 📝 PARTE 2 — Modelo de Texto con Ollama

Usaremos **TinyLlama**, un modelo de lenguaje pequeño pero capaz, ideal para Colab gratuito.

### ¿Cómo funciona Ollama?

```
  Tu código Python
       │
       ▼  petición HTTP POST
  Servidor Ollama   ← corre en localhost:11434
  (ollama serve)
       │
       ▼  carga el modelo
  Modelo TinyLlama  ← archivo .gguf en disco
       │
       ▼  respuesta HTTP JSON
  Tu código Python
```

Primero iniciamos el servidor, luego descargamos el modelo.

In [ ]:
# ============================================================
# CELDA 3 — Iniciar el servidor Ollama en segundo plano
# ============================================================

import subprocess  # Permite lanzar y controlar procesos del sistema desde Python
import time        # Permite pausar la ejecución con time.sleep()
import requests    # Permite hacer peticiones HTTP para verificar que el servidor vive

print('🚀 Iniciando servidor Ollama...')

# subprocess.Popen lanza el proceso en paralelo (no bloquea el notebook).
# Así el servidor queda corriendo en background mientras continuamos.
# DEVNULL descarta los mensajes del servidor para no saturar la salida.
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],           # Equivalente a escribir 'ollama serve' en la terminal
    stdout=subprocess.DEVNULL,     # Descarta la salida estándar del proceso
    stderr=subprocess.DEVNULL      # Descarta los mensajes de error del proceso
)

# Esperamos 4 segundos para que el servidor termine de arrancar
# antes de intentar comunicarnos con él
print('   ⏳ Esperando que el servidor arranque (4 segundos)...')
time.sleep(4)

# Verificamos que el servidor responde haciendo una petición GET simple
try:
    response = requests.get('http://localhost:11434')  # Puerto por defecto de Ollama
    print('✅ Servidor Ollama activo en http://localhost:11434')
except Exception as e:
    # Si falla puede necesitar más tiempo; ejecuta esta celda de nuevo
    print(f'⚠️  El servidor aún no responde. Vuelve a ejecutar esta celda. ({e})')

In [ ]:
# ============================================================
# CELDA 4 — Descargar el modelo de texto TinyLlama
# ============================================================

# 'ollama pull' descarga el modelo desde el repositorio oficial de Ollama.
# TinyLlama pesa ~637 MB. La primera descarga toma 2-5 minutos.
# En ejecuciones siguientes está en caché y es casi instantáneo.

print('📥 Descargando TinyLlama (modelo de texto, ~637 MB)...')
print('   ⏳ Esto puede tomar varios minutos. Por favor espera...')
print()

!ollama pull tinyllama

print()
print('✅ TinyLlama descargado y listo para usar')

In [ ]:
# ============================================================
# CELDA 5 — Definir y probar la función de consulta de texto
# ============================================================

import requests  # Para comunicarnos con el servidor Ollama vía HTTP

def consultar_modelo_texto(prompt: str, modelo: str = 'tinyllama') -> str:
    """
    Envía un prompt al modelo de texto en Ollama y devuelve la respuesta.

    Parámetros:
        prompt : texto que le enviamos al modelo (nuestra pregunta o instrucción)
        modelo : nombre del modelo registrado en Ollama (por defecto 'tinyllama')
    Retorna:
        La respuesta del modelo como cadena de texto
    """

    # Endpoint de la API REST de Ollama para generación de texto
    url = 'http://localhost:11434/api/generate'

    # Cuerpo de la petición en formato diccionario (se enviará como JSON)
    payload = {
        'model': modelo,   # Qué modelo usar
        'prompt': prompt,  # El texto de entrada
        'stream': False    # False = devuelve la respuesta completa de una vez
                           # True  = devolvería tokens uno por uno (streaming)
    }

    try:
        # Petición POST: enviamos el payload y esperamos máximo 120 segundos
        response = requests.post(url, json=payload, timeout=120)

        # raise_for_status() lanza una excepción si el servidor devolvió error HTTP
        response.raise_for_status()

        # Convertimos la respuesta JSON a diccionario y extraemos el campo 'response'
        # .get() retorna None si la clave no existe (evita KeyError)
        data = response.json()
        return data.get('response', 'Sin respuesta')

    except requests.exceptions.Timeout:
        # El modelo tardó más de 2 minutos (prompt muy largo o GPU saturada)
        return '⚠️ El modelo tardó demasiado. Intenta con un prompt más corto.'
    except Exception as e:
        # Capturamos cualquier otro error (servidor caído, red, etc.)
        return f'❌ Error al consultar el modelo: {str(e)}'


# ── Prueba de la función ────────────────────────────────────
print('🧪 Probando el modelo de texto...')
print('   Prompt: "¿Qué es la inteligencia artificial? Responde en 2 oraciones."')
print()

respuesta_prueba = consultar_modelo_texto(
    '¿Qué es la inteligencia artificial? Responde en 2 oraciones.'
)

print('📬 Respuesta del modelo:')
print('-' * 55)
print(respuesta_prueba)
print('-' * 55)
print('✅ Modelo de texto funcionando correctamente')

---
## 🎨 PARTE 3 — Modelo de Generación de Imágenes

Para las imágenes usaremos **Stable Diffusion v1.4** a través de la librería `diffusers` de HuggingFace. Este modelo convierte una descripción de texto en una imagen.

### ¿Cómo funciona Stable Diffusion?

```
  Prompt de texto
       │
       ▼  codificación semántica
  CLIP Text Encoder   ← convierte el texto en vectores numéricos
       │
       ▼  N pasos de refinamiento
  U-Net               ← parte de ruido aleatorio y va refinando
       │               hasta que la imagen coincide con el prompt
       ▼  decodificación
  VAE Decoder         ← convierte el resultado interno en píxeles
       │
       ▼
  Imagen PNG (512x512)
```

> 💡 **Tip:** Stable Diffusion genera mejores imágenes con prompts en **inglés**, ya que fue entrenado principalmente con texto en ese idioma.

In [ ]:
# ============================================================
# CELDA 6 — Instalar librerías para generación de imágenes
# ============================================================

# diffusers   : librería de HuggingFace con pipelines de Stable Diffusion
# transformers: modelos de texto y encoders (CLIP para codificar el prompt)
# accelerate  : optimizaciones de velocidad en GPU (requerido por diffusers)
# torch       : PyTorch, el framework de deep learning que corre los modelos
# torchvision : utilidades de visión computacional para PyTorch

print('📥 Instalando librerías de generación de imágenes...')
!pip install -q diffusers transformers accelerate torch torchvision
print('✅ Librerías de imagen instaladas correctamente')

In [ ]:
# ============================================================
# CELDA 7 — Cargar Stable Diffusion en memoria de GPU
# ============================================================

import torch                                     # Framework de deep learning
from diffusers import StableDiffusionPipeline    # Pipeline completo de generación de imágenes
from PIL import Image                            # Para manipular imágenes (librería Pillow)

print('🔍 Detectando hardware disponible...')

# torch.cuda.is_available() devuelve True si hay una GPU NVIDIA disponible.
# En Colab con T4 activa, esto debería ser True.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'   🖥️  Dispositivo detectado: {device.upper()}')

if device == 'cpu':
    print('   ⚠️  Sin GPU: la generación de imágenes será muy lenta (10+ minutos).')
    print('      Cambia a GPU T4 en: Entorno de ejecución → Cambiar tipo → GPU')

# float16 usa la mitad de la memoria que float32,
# lo que permite cargar Stable Diffusion en la GPU de Colab (16 GB VRAM).
# En CPU debemos usar float32 porque no soporta operaciones float16.
dtype = torch.float16 if device == 'cuda' else torch.float32

print()
print('📥 Cargando Stable Diffusion v1.4 (~4 GB en disco)...')
print('   ⏳ Primera carga puede tomar entre 3 y 8 minutos...')
print()

# from_pretrained descarga el modelo desde HuggingFace Hub la primera vez
# y lo carga desde caché local en ejecuciones posteriores.
pipeline_imagen = StableDiffusionPipeline.from_pretrained(
    'CompVis/stable-diffusion-v1-4',   # Identificador del modelo en HuggingFace
    torch_dtype=dtype,                  # Precisión numérica (float16 ahorra VRAM)
    safety_checker=None,                # Desactivamos el filtro de contenido
    requires_safety_checker=False       # para simplificar la actividad académica
)

# .to(device) mueve todos los pesos del modelo a la GPU (o CPU si no hay GPU)
pipeline_imagen = pipeline_imagen.to(device)

# enable_attention_slicing procesa la imagen en partes más pequeñas
# para reducir el consumo de VRAM y evitar errores 'out of memory'
if device == 'cuda':
    try:
        pipeline_imagen.enable_attention_slicing()
        print('   ⚡ Optimización de memoria GPU activada (attention slicing)')
    except Exception:
        pass  # Si no está disponible, continuamos sin ella

print()
print('✅ Modelo de imágenes cargado y listo en', device.upper())

In [ ]:
# ============================================================
# CELDA 8 — Definir y probar la función de generación de imágenes
# ============================================================

import torch
from PIL import Image
from IPython.display import display  # Para mostrar imágenes dentro del notebook Colab

def generar_imagen(prompt: str, pasos: int = 20) -> Image.Image:
    """
    Genera una imagen a partir de una descripción de texto.

    Parámetros:
        prompt : descripción de la imagen a generar (inglés recomendado)
        pasos  : número de pasos de difusión.
                 Más pasos = mejor calidad, pero más tiempo.
                 Rango recomendado: 15-50.
    Retorna:
        Objeto PIL.Image con la imagen generada (512x512 px por defecto)
    """

    # Mostramos el prompt truncado si es muy largo
    preview = prompt[:60] + '...' if len(prompt) > 60 else prompt
    print(f'🎨 Generando imagen para: "{preview}"')
    print(f'   Pasos de difusión: {pasos}')

    # torch.autocast activa la precisión mixta automáticamente:
    # usa float16 donde es seguro para acelerar el cómputo,
    # y float32 donde la precisión es crítica.
    dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
    with torch.autocast(dispositivo):
        resultado = pipeline_imagen(
            prompt,                     # Descripción de la imagen
            num_inference_steps=pasos,  # Número de pasos de refinamiento
            guidance_scale=7.5          # Fidelidad al prompt: 7.5 es el estándar.
                                        # Más alto = más fiel pero menos creativo.
        )

    # El pipeline devuelve un objeto con una lista de imágenes generadas.
    # Tomamos la primera (índice 0) ya que pedimos solo una imagen.
    imagen = resultado.images[0]
    return imagen


# ── Prueba de la función ────────────────────────────────────
print('🧪 Probando el modelo de imágenes...')
print('   Prompt: "a robot reading a book in a library, digital art"')
print()

imagen_prueba = generar_imagen(
    'a robot reading a book in a library, digital art',
    pasos=20  # 20 pasos: buen balance entre calidad y velocidad
)

print()
print('🖼️  Imagen generada:')
display(imagen_prueba)  # Muestra la imagen directamente en el notebook
print('✅ Modelo de imágenes funcionando correctamente')

---
## 💻 PARTE 4 — Interfaz Gradio Unificada

Gradio nos permite construir una interfaz web completa en Python. La interfaz tendrá:

- Un **selector de modo** (Texto o Imagen)
- Un **campo de texto** para escribir el prompt
- Una **salida dinámica** que muestra texto o imagen según el modo

### Arquitectura de la interfaz

```
  Usuario escribe prompt
         │
         ▼
  procesar_solicitud()   ← función router central
    ├── modo Texto  ──► consultar_modelo_texto()  ──► muestra texto
    └── modo Imagen ──► generar_imagen()          ──► muestra imagen
```

In [ ]:
# ============================================================
# CELDA 9 — Función router: dirige al modelo correcto
# ============================================================

from PIL import Image
from typing import Tuple, Optional  # Para anotaciones de tipo en la firma de la función

def procesar_solicitud(
    modo: str,         # Modo seleccionado en la interfaz: Texto o Imagen
    prompt: str,       # Texto escrito por el usuario
    pasos_imagen: int  # Pasos de difusión (solo relevante para el modo Imagen)
) -> Tuple[Optional[str], Optional[Image.Image]]:
    """
    Función router: recibe la solicitud del usuario y la dirige
    al modelo correcto según el modo seleccionado.

    Retorna una tupla (texto, imagen) donde siempre uno de
    los dos es None según el modo activo:
      modo Texto  → (respuesta_texto, None)
      modo Imagen → (None, imagen_PIL)
    """

    # Validación de entrada: no procesamos si el campo está vacío
    if not prompt or prompt.strip() == '':
        return '⚠️ Por favor escribe un prompt antes de enviar.', None

    # .strip() elimina espacios en blanco al inicio y al final
    prompt = prompt.strip()

    # ── Rama TEXTO ──────────────────────────────────────────
    if modo == '📝 Texto (TinyLlama)':
        print(f'[Router] → Modo TEXTO | Prompt: "{prompt[:45]}..."')

        # Llamamos a la función definida en Celda 5
        respuesta = consultar_modelo_texto(prompt)

        # Primer valor = texto, segundo valor = None (no hay imagen)
        return respuesta, None

    # ── Rama IMAGEN ─────────────────────────────────────────
    elif modo == '🎨 Imagen (Stable Diffusion)':
        print(f'[Router] → Modo IMAGEN | Prompt: "{prompt[:45]}..."')

        # Llamamos a la función definida en Celda 8
        imagen = generar_imagen(prompt, pasos=int(pasos_imagen))

        # Primer valor = None (no hay texto), segundo valor = imagen
        return None, imagen

    # ── Modo desconocido (no debería ocurrir nunca) ──────────
    else:
        return '❌ Modo no reconocido. Selecciona Texto o Imagen.', None


print('✅ Función router definida')
print('   Rutas disponibles:')
print('   • 📝 Texto (TinyLlama)         → retorna (texto, None)')
print('   • 🎨 Imagen (Stable Diffusion) → retorna (None, imagen)')

In [ ]:
# ============================================================
# CELDA 10 — Construir la interfaz Gradio
# ============================================================

import gradio as gr  # Framework para crear interfaces web con Python

# ── CSS personalizado ────────────────────────────────────────
# Gradio acepta estilos CSS para personalizar la apariencia visual.
# Aquí definimos el ancho máximo de la interfaz y el estilo del banner.
css = """
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
    max-width: 820px;
    margin: 0 auto;
}
.titulo {
    text-align: center;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    color: white;
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 10px;
}
"""

# ── Construcción con gr.Blocks ───────────────────────────────
# gr.Blocks() es el modo avanzado de Gradio que permite organizar
# componentes libremente y conectar eventos complejos.
# El 'with' asegura que todos los componentes dentro pertenezcan a este Blocks.
with gr.Blocks(css=css, title='Asistente IA Multimodal') as demo:

    # ── Encabezado HTML personalizado ───────────────────────
    # gr.HTML() inserta código HTML plano dentro de la interfaz Gradio
    gr.HTML("""
    <div class="titulo">
        <h1>🤖 Asistente IA Multimodal</h1>
        <p>Powered by <b>Ollama + TinyLlama</b> y <b>Stable Diffusion v1.4</b></p>
    </div>
    """)

    # ── Selector de modo ─────────────────────────────────────
    # gr.Row() organiza los componentes internos en una fila horizontal
    with gr.Row():
        # gr.Radio muestra opciones de selección única (botones de radio)
        modo = gr.Radio(
            choices=['📝 Texto (TinyLlama)', '🎨 Imagen (Stable Diffusion)'],
            value='📝 Texto (TinyLlama)',   # Opción activa por defecto al cargar
            label='🔀 Selecciona el modelo',
            info='Elige si quieres generar texto o una imagen'
        )

    # ── Panel de configuración colapsable ────────────────────
    # gr.Accordion crea una sección que el usuario puede expandir o colapsar
    with gr.Accordion('⚙️ Configuración avanzada (solo para imagen)', open=False):
        # gr.Slider crea un control deslizante numérico
        pasos = gr.Slider(
            minimum=10,   # Valor mínimo permitido
            maximum=50,   # Valor máximo permitido
            value=20,     # Valor inicial
            step=5,       # Cuánto sube/baja con cada movimiento del slider
            label='Pasos de difusión',
            info='Más pasos = mejor calidad, pero más tiempo de espera'
        )
        # gr.Markdown muestra texto con formato dentro de la interfaz
        gr.Markdown("""
        > 💡 **Tip:** Escribe los prompts de imagen en **inglés** para mejores resultados.
        > Ejemplo: *"a golden retriever sitting on a mountain, sunset, photorealistic"*
        """)

    # ── Campo de entrada del usuario ─────────────────────────
    with gr.Row():
        # gr.Textbox muestra un área de texto editable por el usuario
        prompt_input = gr.Textbox(
            label='✏️ Escribe tu prompt aquí',
            placeholder='Escribe tu pregunta o descripción de imagen...',
            lines=3,   # Altura inicial del área de texto (en líneas)
            scale=4    # Ancho relativo dentro de la fila (ocupa 4 unidades)
        )

    # ── Botones de acción ────────────────────────────────────
    with gr.Row():
        # gr.ClearButton limpia automáticamente los componentes conectados
        btn_limpiar = gr.ClearButton(
            value='🗑️ Limpiar',
            scale=1   # Ocupa 1 unidad de ancho (más angosto)
        )
        # gr.Button crea un botón interactivo
        btn_enviar = gr.Button(
            value='🚀 Generar',
            variant='primary',  # Estilo visual destacado (azul/color primario)
            scale=3             # Ocupa 3 unidades (más ancho que Limpiar)
        )

    # ── Área de salida: Texto ────────────────────────────────
    # Visible por defecto porque el modo inicial es Texto
    salida_texto = gr.Textbox(
        label='📄 Respuesta de texto',
        lines=8,
        visible=True,             # Se muestra al cargar la interfaz
        show_copy_button=True     # Añade botón para copiar el texto al portapapeles
    )

    # ── Área de salida: Imagen ───────────────────────────────
    # Oculta por defecto; se mostrará al cambiar al modo Imagen
    salida_imagen = gr.Image(
        label='🖼️ Imagen generada',
        visible=False,            # Oculta al cargar la interfaz
        type='pil',               # Acepta objetos PIL.Image (los que retorna el modelo)
        show_download_button=True # Permite al usuario descargar la imagen generada
    )

    # ── Indicador de estado ──────────────────────────────────
    # Se actualiza dinámicamente después de cada generación
    estado = gr.Markdown('_Listo. Escribe un prompt y presiona Generar._')

    # ── Ejemplos de prompts precargados ──────────────────────
    # gr.Examples muestra tarjetas de ejemplo clicables.
    # Al hacer clic, los valores se cargan automáticamente en los inputs indicados.
    gr.Examples(
        examples=[
            ['📝 Texto (TinyLlama)', 'Explica qué es el aprendizaje automático en términos simples.', 20],
            ['📝 Texto (TinyLlama)', 'Write a short poem about technology and nature.', 20],
            ['📝 Texto (TinyLlama)', '¿Cuáles son las diferencias entre Python y JavaScript?', 20],
            ['🎨 Imagen (Stable Diffusion)', 'a futuristic city at night with neon lights, cyberpunk style', 25],
            ['🎨 Imagen (Stable Diffusion)', 'a cute cat wearing a graduation hat, digital art', 20],
            ['🎨 Imagen (Stable Diffusion)', 'a peaceful mountain lake at sunrise, photorealistic', 30],
        ],
        inputs=[modo, prompt_input, pasos],   # Los valores de cada fila van a estos componentes
        label='📚 Ejemplos de prompts (haz clic para cargar)'
    )

    # ============================================================
    # LÓGICA DE EVENTOS
    # Los eventos conectan acciones del usuario con funciones Python.
    # Gradio llamará a estas funciones automáticamente.
    # ============================================================

    def actualizar_visibilidad(modo_seleccionado):
        """
        Se ejecuta cada vez que el usuario cambia el selector de modo.
        Muestra el componente de salida correcto y oculta el otro.
        También actualiza el placeholder del campo de texto.
        """
        # Determinamos si el modo activo es Imagen
        es_imagen = (modo_seleccionado == '🎨 Imagen (Stable Diffusion)')

        # gr.update() modifica propiedades de un componente ya renderizado
        # sin necesidad de recrearlo. Retornamos uno por cada output declarado.
        return (
            gr.update(visible=not es_imagen),  # salida_texto: visible en modo Texto
            gr.update(visible=es_imagen),       # salida_imagen: visible en modo Imagen
            gr.update(
                placeholder=(
                    'Describe la imagen que quieres generar (en inglés es mejor)...'
                    if es_imagen else
                    'Escribe tu pregunta o instrucción...'
                )
            )  # prompt_input: cambia el texto de ayuda según el modo
        )

    def ejecutar_modelo(modo_sel, prompt_text, pasos_val):
        """
        Se ejecuta al presionar Generar o al teclear Enter.
        Llama al router y devuelve los resultados a los componentes de salida.
        """
        # Validamos que hay texto antes de procesar
        if not prompt_text or not prompt_text.strip():
            return '⚠️ Por favor escribe un prompt primero.', None, '_⚠️ Campo vacío._'

        # Llamamos a la función router definida en Celda 9
        texto_out, imagen_out = procesar_solicitud(
            modo=modo_sel,
            prompt=prompt_text,
            pasos_imagen=int(pasos_val)  # Convertimos a int para mayor seguridad
        )

        # Construimos el mensaje de estado descriptivo
        if imagen_out is not None:
            estado_msg = f'✅ Imagen generada con {int(pasos_val)} pasos de difusión.'
        else:
            palabras = len(texto_out.split()) if texto_out else 0
            estado_msg = f'✅ Respuesta generada ({palabras} palabras).'

        # Retornamos los tres outputs en el mismo orden que están declarados:
        # 1) salida_texto, 2) salida_imagen, 3) estado
        return texto_out or '', imagen_out, estado_msg

    # ── Registrar el evento de cambio de modo ────────────────
    # .change() se activa cuando el valor del componente cambia.
    # fn     : función a llamar
    # inputs : componentes cuyos valores se leen y pasan como argumentos
    # outputs: componentes que reciben los valores retornados por la función
    modo.change(
        fn=actualizar_visibilidad,
        inputs=[modo],
        outputs=[salida_texto, salida_imagen, prompt_input]
    )

    # ── Registrar el evento del botón Generar ────────────────
    # .click() se activa cuando el usuario hace clic en el botón
    btn_enviar.click(
        fn=ejecutar_modelo,
        inputs=[modo, prompt_input, pasos],
        outputs=[salida_texto, salida_imagen, estado]
    )

    # ── Registrar el evento de Enter en el Textbox ───────────
    # .submit() se activa cuando el usuario presiona Enter dentro del campo de texto
    # (comportamiento idéntico al clic en Generar)
    prompt_input.submit(
        fn=ejecutar_modelo,
        inputs=[modo, prompt_input, pasos],
        outputs=[salida_texto, salida_imagen, estado]
    )

    # ── Conectar el botón Limpiar ─────────────────────────────
    # .add() le dice al ClearButton qué componentes debe limpiar al hacer clic
    btn_limpiar.add([prompt_input, salida_texto, salida_imagen])


# ── Lanzar la interfaz ───────────────────────────────────────
# share=True genera un link público temporal válido 72 horas.
# En la Parte 7 lo reemplazaremos por Ngrok para mayor estabilidad.
print('🚀 Lanzando interfaz Gradio...')
print('   Haz clic en el enlace "Running on public URL" que aparece abajo 👇')
print()

demo.launch(
    share=True,       # Publica la app en un link temporal de gradio.live
    debug=False,      # False: no muestra trazas de error en pantalla
    show_error=True   # True: muestra mensajes de error simplificados en la UI
)

---
## 🧪 PARTE 5 — Reflexión

### ✍️ Responde las siguientes preguntas en la celda de texto debajo

### 📝 Mis Respuestas

*Reemplaza este texto con tus respuestas*

---

**1. ¿Qué diferencia fundamental existe entre TinyLlama y Stable Diffusion? ¿Con qué tipo de datos fue entrenado cada uno?**

> *Tu respuesta aquí...*

---

**2. En el código de Gradio, ¿por qué usamos `gr.update(visible=True/False)` al cambiar de modo? ¿Qué pasaría si no lo hiciéramos?**

> *Tu respuesta aquí...*

---

**3. ¿Por qué Stable Diffusion genera mejores imágenes con prompts en inglés? ¿Tiene esto implicaciones de accesibilidad?**

> *Tu respuesta aquí...*

---

**4. ¿Qué ventajas tiene correr modelos localmente con Ollama versus usar una API en la nube como OpenAI?**

> *Tu respuesta aquí...*

---
## 🏆 PARTE 6 — Retos Opcionales

### Reto A — Historial de conversación
Modifica la interfaz para que el modo texto recuerde el historial. Reemplaza `gr.Textbox()` por `gr.Chatbot()` y adapta la función de envío para pasar el historial completo a Ollama en cada petición.

### Reto B — Streaming de texto
Cambia `'stream': False` a `True` en `consultar_modelo_texto` y usa `yield` en la función de Gradio para que el texto aparezca token por token (como ChatGPT).

### Reto C — Traducción automática del prompt
Antes de enviar a Stable Diffusion, traduce el prompt al inglés usando `deep_translator`. Así el usuario puede escribir en español y obtener imágenes de buena calidad.

In [ ]:
# ============================================================
# CELDA DE RETO — Espacio libre para tus experimentos
# ============================================================

# Escribe aquí tu solución al reto que elegiste.
# No hay respuesta incorrecta: lo importante es experimentar.

# Tu código aquí...


---
## 🌐 PARTE 7 — Publicación con Ngrok

El `share=True` de Gradio genera un link público, pero caduca en 72 horas y es inestable. **Ngrok** crea un túnel HTTPS permanente mientras Colab esté activo, ideal para compartir la app con toda la clase en tiempo real.

### ¿Cómo funciona?

```
  COLAB (localhost:7860)  ──►  TÚNEL NGROK  ──►  https://xxxx.ngrok-free.app
         Gradio app               HTTPS              URL pública compartible
```

### Pasos previos — Obtener tu token de Ngrok

1. Crea una cuenta gratuita en [https://ngrok.com](https://ngrok.com)
2. Ve a **Your Authtoken**: [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Copia tu token (se ve así: `2abc123XYZ_xxxxxxxx`)
4. En Colab, haz clic en el ícono 🔑 **Secrets** (panel izquierdo), crea un secret llamado `NGROK_TOKEN` y pega tu token como valor

> 🔐 Guardar el token en Secrets evita que quede visible en el código del notebook al compartirlo.

In [ ]:
# ============================================================
# CELDA 11 — Instalar pyngrok (cliente Python de Ngrok)
# ============================================================

# pyngrok es la librería oficial de Python para controlar Ngrok:
# abrir túneles, cerrarlos, listarlos y configurarlos desde código.
!pip install -q pyngrok

# Importamos los dos módulos que usaremos:
# ngrok : control de túneles (connect, kill, get_tunnels)
# conf  : configuración global de Ngrok (auth_token, región, etc.)
from pyngrok import ngrok, conf

print('✅ pyngrok instalado correctamente')

In [ ]:
# ============================================================
# CELDA 12 — Autenticar Ngrok con tu token
# ============================================================

from pyngrok import ngrok, conf

# ── Opción A (recomendada): leer desde Secrets de Colab ─────
# userdata.get() lee el valor guardado en el panel de Secrets.
# Esto evita exponer el token en el código del notebook.
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')  # Lee el secret 'NGROK_TOKEN'
    print('✅ Token leído desde Secrets de Colab')
except Exception:
    # ── Opción B: pegar el token directamente ────────────────
    # Úsala solo si no puedes usar Secrets.
    # Importante: no compartas el notebook con el token escrito aquí.
    NGROK_TOKEN = 'PEGA_AQUI_TU_TOKEN_DE_NGROK'  # ← reemplaza esto
    print('⚠️  Usando token escrito en el código (Opción B)')

# Verificamos que el token no es el placeholder antes de continuar
if not NGROK_TOKEN or NGROK_TOKEN == 'PEGA_AQUI_TU_TOKEN_DE_NGROK':
    raise ValueError(
        '❌ Token de Ngrok no configurado.\n'
        '   Obtén uno gratis en: https://dashboard.ngrok.com/get-started/your-authtoken\n'
        '   Luego guárdalo en Colab Secrets con el nombre: NGROK_TOKEN'
    )

# conf.get_default().auth_token registra el token globalmente
# para que todos los túneles que abramos estén autenticados
conf.get_default().auth_token = NGROK_TOKEN
print('✅ Ngrok autenticado correctamente')

In [ ]:
# ============================================================
# CELDA 13 — Relanzar Gradio con túnel Ngrok
# ============================================================

from pyngrok import ngrok
import gradio as gr

GRADIO_PORT = 7860  # Puerto por defecto donde Gradio levanta su servidor web

# Cerramos la instancia de Gradio anterior (si existe) para liberar el puerto.
# Si no existía, el bloque except lo maneja silenciosamente.
try:
    demo.close()
    print('🔄 Instancia anterior de Gradio cerrada')
except Exception:
    pass

# Cerramos todos los túneles Ngrok previos.
# La cuenta gratuita permite solo 1 túnel activo a la vez.
ngrok.kill()

# ── Paso 1: Abrir el túnel ANTES de lanzar Gradio ───────────
# ngrok.connect(puerto, protocolo) crea un túnel que redirige
# el tráfico público entrante al puerto local indicado
print('🌐 Abriendo túnel Ngrok...')
tunel = ngrok.connect(GRADIO_PORT, 'http')

# .public_url contiene la URL HTTPS pública generada por Ngrok
url_publica = tunel.public_url

print()
print('=' * 58)
print(f'  🔗 URL PÚBLICA:  {url_publica}')
print('=' * 58)
print('  Comparte este enlace con tus compañeros de clase.')
print('  Válido mientras esta sesión de Colab esté activa.')
print()

# ── Paso 2: Lanzar Gradio en el mismo puerto ─────────────────
# server_name='0.0.0.0' hace que Gradio escuche en todas las interfaces
# de red del sistema (no solo localhost), necesario para que Ngrok
# pueda actuar como proxy hacia él.
print('🚀 Iniciando interfaz Gradio...')
demo.launch(
    server_name='0.0.0.0',    # Acepta conexiones desde cualquier IP
    server_port=GRADIO_PORT,  # Mismo puerto que usa el túnel Ngrok
    share=False,              # False: Ngrok reemplaza el share de Gradio
    inline=False,             # False: no intenta renderizar en el notebook
    quiet=True                # True: suprime los mensajes de inicio de Gradio
)

print()
print('✅ Todo listo. Abre en tu navegador:')
print(f'   👉  {url_publica}')

In [ ]:
# ============================================================
# CELDA 14 — Verificar el estado del túnel activo
# ============================================================

from pyngrok import ngrok

# ngrok.get_tunnels() retorna una lista con todos los túneles activos
tuneles_activos = ngrok.get_tunnels()

if not tuneles_activos:
    print('⚠️  No hay túneles activos. Ejecuta la Celda 13 primero.')
else:
    print(f'📡 Túneles Ngrok activos: {len(tuneles_activos)}')
    print('-' * 50)
    for t in tuneles_activos:
        print(f'  🔗 URL pública  : {t.public_url}')       # Link que usan los usuarios externos
        print(f'  🏠 Puerto local : {t.config["addr"]}')   # Puerto interno al que redirige
        print(f'  📋 Protocolo    : {t.proto}')            # http o https
        print('-' * 50)

# El dashboard local de Ngrok muestra métricas en tiempo real:
# peticiones recibidas, latencia, respuestas, errores.
# Solo es accesible desde dentro de Colab (no desde tu navegador local).
print()
print('📊 Dashboard de métricas (solo desde Colab): http://localhost:4040')

In [ ]:
# ============================================================
# CELDA 15 — (OPCIONAL) Usar un dominio estático de Ngrok
# ============================================================

# Con la cuenta gratuita, el subdominio cambia cada vez que
# reinicias Colab. Ngrok Free permite reservar UN dominio
# estático gratuito que siempre será el mismo.
#
# Pasos para obtenerlo:
# 1. Ve a: https://dashboard.ngrok.com/cloud-edge/domains
# 2. Haz clic en "New Domain" (te asignan uno gratis automáticamente)
# 3. Copia el dominio (ej: "mi-app-ia.ngrok-free.app")
# 4. Pégalo en la variable DOMINIO_ESTATICO abajo

from pyngrok import ngrok

DOMINIO_ESTATICO = 'TU-DOMINIO.ngrok-free.app'  # ← reemplaza con el tuyo

# Verificamos que fue reemplazado el placeholder antes de continuar
if DOMINIO_ESTATICO == 'TU-DOMINIO.ngrok-free.app':
    print('⚠️  Esta celda es opcional.')
    print('   Para activarla, reemplaza DOMINIO_ESTATICO con tu dominio reservado.')
    print('   Ve a: https://dashboard.ngrok.com/cloud-edge/domains')
else:
    # Cerramos el túnel aleatorio anterior para abrir el fijo
    ngrok.kill()

    # El parámetro domain= le indica a Ngrok que use ese subdominio
    # en lugar de generar uno aleatorio nuevo
    tunel_fijo = ngrok.connect(
        7860,
        'http',
        domain=DOMINIO_ESTATICO   # Subdominio reservado en tu cuenta Ngrok
    )
    print('✅ Túnel con dominio estático activo:')
    print(f'   🔗 https://{DOMINIO_ESTATICO}')
    print('   Este enlace no cambiará al reiniciar la sesión.')

### 💡 Comparación: Gradio `share=True` vs Ngrok

| Característica | `share=True` de Gradio | Ngrok (gratuito) |
|---|---|---|
| **Configuración** | Automática, cero config | Requiere cuenta + token |
| **Duración del link** | 72 horas máximo | Mientras Colab esté activo |
| **Subdominio** | Aleatorio, cambia siempre | Aleatorio (o fijo con dominio estático) |
| **HTTPS** | ✅ Sí | ✅ Sí |
| **Dashboard de tráfico** | ❌ No | ✅ Sí (localhost:4040) |
| **Control del túnel** | ❌ Limitado | ✅ Total (abrir/cerrar por código) |
| **Límite de conexiones** | Limitado por Gradio | 40 conn/min (plan free) |
| **Ideal para** | Demos rápidas | Compartir con grupos o clases |

> 🎓 Para compartir con toda la clase durante la sesión, **Ngrok es la mejor opción**.

---
## 🧹 Limpieza Final

Ejecuta esta celda al terminar la actividad para liberar todos los recursos de Colab.

In [ ]:
# ============================================================
# CELDA FINAL — Liberar todos los recursos
# ============================================================

import torch
from pyngrok import ngrok

# Cerramos el túnel Ngrok: libera la conexión en los servidores de Ngrok
# y deja de aceptar tráfico externo
try:
    ngrok.kill()   # Termina todos los túneles activos a la vez
    print('✅ Túnel Ngrok cerrado')
except Exception:
    pass

# Cerramos el servidor web de Gradio: libera el puerto 7860
try:
    demo.close()
    print('✅ Interfaz Gradio cerrada')
except Exception:
    pass

# Terminamos el proceso de Ollama: libera la CPU/GPU que usaba
# .terminate() envía la señal SIGTERM al proceso para que se cierre
try:
    ollama_process.terminate()
    print('✅ Servidor Ollama detenido')
except Exception:
    pass

# Eliminamos el pipeline de Stable Diffusion de la memoria RAM/VRAM
try:
    del pipeline_imagen          # Elimina el objeto Python y sus referencias
    torch.cuda.empty_cache()     # Vacía la caché de VRAM reservada por PyTorch
    print('✅ Memoria GPU liberada')
except Exception:
    pass

print()
print('🏁 Sesión finalizada correctamente. ¡Buen trabajo!')

---

## 📊 Rúbrica de Evaluación

| Criterio | Excelente (5) | Satisfactorio (3) | En desarrollo (1) |
|----------|--------------|-------------------|-------------------|
| **Instalación y configuración** | Todos los modelos corren sin errores | Al menos un modelo corre | No logra levantar ningún modelo |
| **Interfaz funcional** | Gradio responde correctamente a ambos modos | Un solo modo funciona | La interfaz no carga |
| **Calidad de prompts probados** | 5+ prompts variados y creativos | 3-4 prompts básicos | Solo usa los ejemplos provistos |
| **Publicación con Ngrok** | URL pública funcionando y compartida | Ngrok instalado pero con errores | No intenta la publicación |
| **Reflexión escrita** | Respuestas detalladas con ejemplos propios | Respuestas incompletas | Sin respuestas |
| **Reto opcional** | Implementa y explica al menos un reto | Intenta un reto parcialmente | No intenta los retos |

---

> 📎 **Entrega:** Descarga este notebook (`.ipynb`) con todas las celdas ejecutadas y las respuestas de reflexión completadas. Incluye una captura de pantalla de tu URL de Ngrok funcionando. Súbelo al aula virtual antes de la fecha límite.

---
*Actividad diseñada para el aprendizaje de IA Generativa con herramientas de código abierto.*